## 1. Setup, imports and configuration
Installs Ultralytics, imports everything, and defines the settings: dataset path, box type, image size, and the list of models to benchmark. All models are used **as downloaded (COCO-pretrained), with no finetuning**.

`BOX_TYPE = "vbox"` (visible body) is the best match for COCO's `person` class. Use `"fbox"` for full-body boxes.

In [ ]:
!pip install -U ultralytics -q

import json, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO, RTDETR

# ----------------------------- CONFIG ----------------------------------------
SRC      = Path("/kaggle/input/datasets/loctran0941/crowdhuman/CrowdHuman")
WORK     = Path("/kaggle/working")
DS       = WORK / "crowdhuman_val_yolo"
RESULTS  = WORK / "results_zeroshot.json"

BOX_TYPE = "vbox"      # "vbox" visible body | "fbox" full body | "hbox" head
IMGSZ    = 640
BATCH    = 16
DEVICE   = 0

MODELS = [
    ("YOLOv8n", "yolov8n.pt", YOLO), ("YOLOv8s", "yolov8s.pt", YOLO),
    ("YOLOv8m", "yolov8m.pt", YOLO), ("YOLOv8l", "yolov8l.pt", YOLO),
    ("YOLO11n", "yolo11n.pt", YOLO), ("YOLO11s", "yolo11s.pt", YOLO),
    ("YOLO11m", "yolo11m.pt", YOLO), ("YOLO11l", "yolo11l.pt", YOLO),
    ("YOLO12n", "yolo12n.pt", YOLO), ("YOLO12s", "yolo12s.pt", YOLO),
    ("YOLO12m", "yolo12m.pt", YOLO), ("YOLO12l", "yolo12l.pt", YOLO),
    ("YOLO26n", "yolo26n.pt", YOLO), ("YOLO26s", "yolo26s.pt", YOLO),
    ("YOLO26m", "yolo26m.pt", YOLO), ("YOLO26l", "yolo26l.pt", YOLO),
    ("RT-DETR-l", "rtdetr-l.pt", RTDETR), ("RT-DETR-x", "rtdetr-x.pt", RTDETR),
]

## 2. Convert the CrowdHuman val set to YOLO format
Reads `annotation_val.odgt`, finds the images recursively (so the sub-folder layout doesn't matter), and writes YOLO-format labels (one class: `person`). Images are symlinked because `/kaggle/input` is read-only. `mask` boxes and boxes flagged `ignore` are skipped. Only the 4,370 val images are used.

In [ ]:
def prepare_val():
    n_lbl = len(list((DS / "labels" / "val").glob("*.txt"))) if DS.exists() else 0
    if (DS / "data.yaml").exists() and n_lbl > 0:
        return str(DS / "data.yaml")
    shutil.rmtree(DS, ignore_errors=True)

    img_dst = DS / "images" / "val"; lbl_dst = DS / "labels" / "val"
    img_dst.mkdir(parents=True); lbl_dst.mkdir(parents=True)

    exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
    index = {p.stem: p for p in (SRC / "val").rglob("*") if p.suffix in exts}
    print(f"found {len(index)} val images")
    if not index:
        raise RuntimeError(f"No images under {SRC/'val'}. Check: !find {SRC} -maxdepth 3 | head -30")

    with open(SRC / "annotation_val.odgt") as f:
        items = [json.loads(l) for l in f if l.strip()]

    n_img = n_box = 0
    for it in tqdm(items, desc="convert val"):
        src = index.get(it["ID"])
        if src is None:
            continue
        (img_dst / src.name).symlink_to(src)
        W, H = Image.open(src).size
        rows = []
        for g in it["gtboxes"]:
            if g["tag"] != "person" or g.get("extra", {}).get("ignore", 0) == 1:
                continue
            x, y, w, h = g[BOX_TYPE]
            x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
            if x2 - x1 < 2 or y2 - y1 < 2:
                continue
            rows.append(f"0 {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
        (lbl_dst / f"{it['ID']}.txt").write_text("\n".join(rows))
        n_img += 1; n_box += len(rows)
    print(f"val: {n_img} images, {n_box} boxes")
    if n_img == 0:
        raise RuntimeError("0 images matched the annotations.")

    # 'train' key is only there because Ultralytics requires it; it is never used.
    (DS / "data.yaml").write_text(
        f"path: {DS}\ntrain: images/val\nval: images/val\nnames:\n  0: person\n")
    return str(DS / "data.yaml")

data_yaml = prepare_val()
print("data yaml:", data_yaml)

## 3. Benchmark functions (accuracy, speed, memory)
- `gpu_speed`: pure model forward pass, batch 1, FP16, timed with CUDA events (median of 200 runs after warm-up). Gives GPU inference (ms), FPS and peak GPU memory (GB).
- `run_one`: loads the pretrained weights, counts parameters, evaluates on the val set for Precision / Recall / mAP@50 / mAP@50:95 (`classes=[0]` keeps only `person` predictions), then runs the speed test.

In [ ]:
@torch.no_grad()
def gpu_speed(weights, cls, n_warmup=30, n_iter=200):
    """Pure model forward, batch=1, FP16, CUDA events -> (ms, fps, peak_GB)."""
    m = cls(weights)
    net = m.model.fuse() if hasattr(m.model, "fuse") else m.model
    net = net.half().cuda().eval()
    x = torch.rand(1, 3, IMGSZ, IMGSZ, device="cuda", dtype=torch.half)

    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    for _ in range(n_warmup):
        net(x)
    torch.cuda.synchronize()

    times = []
    for _ in range(n_iter):
        s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        s.record(); net(x); e.record(); torch.cuda.synchronize()
        times.append(s.elapsed_time(e))
    ms = float(np.median(times))
    peak = torch.cuda.max_memory_allocated() / 1024**3
    del net, m; torch.cuda.empty_cache()
    return ms, 1000.0 / ms, peak


def run_one(name, weights, cls, data_yaml):
    m = cls(weights)                                   # COCO weights, no training
    params = sum(p.numel() for p in m.model.parameters()) / 1e6
    r = m.val(data=data_yaml, split="val", imgsz=IMGSZ, batch=BATCH,
              conf=0.001, iou=0.6, device=DEVICE,
              classes=[0],                             # keep only COCO 'person'
              plots=False, verbose=False)
    del m; torch.cuda.empty_cache()

    ms, fps, peak = gpu_speed(weights, cls)
    return {
        "Detector": name,
        "Params (M)": round(params, 2),
        "Precision": round(float(r.box.mp), 4),
        "Recall": round(float(r.box.mr), 4),
        "mAP@50": round(float(r.box.map50), 4),
        "mAP@50:95": round(float(r.box.map), 4),
        "GPU inference (ms)": round(ms, 2),
        "GPU FPS": round(fps, 2),
        "Peak GPU (GB)": round(peak, 2),
    }

## 4. Table renderer
Draws the results DataFrame as an image with the same layout as the reference table: arrows in the headers, zebra rows, and the best mAP first.

In [ ]:
def render_table(df, path):
    import matplotlib.pyplot as plt
    heads = {
        "Detector": "Detector", "Params (M)": "Params\n(M)",
        "Precision": "Precision\n↑", "Recall": "Recall\n↑",
        "mAP@50": "mAP@50\n↑", "mAP@50:95": "mAP@50:95\n↑",
        "GPU inference (ms)": "GPU\ninference\n(ms) ↓", "GPU FPS": "GPU\nFPS ↑",
        "Peak GPU (GB)": "Peak\nGPU\n(GB) ↓",
    }
    fmt = {"Params (M)": "{:.2f}", "Precision": "{:.4f}", "Recall": "{:.4f}",
           "mAP@50": "{:.4f}", "mAP@50:95": "{:.4f}", "GPU inference (ms)": "{:.2f}",
           "GPU FPS": "{:.2f}", "Peak GPU (GB)": "{:.2f}"}
    cells = [[r["Detector"]] + [fmt[c].format(r[c]) for c in df.columns[1:]]
             for _, r in df.iterrows()]

    fig, ax = plt.subplots(figsize=(11, 0.38 * (len(df) + 3)))
    ax.axis("off")
    t = ax.table(cellText=cells, colLabels=[heads[c] for c in df.columns],
                 loc="center", cellLoc="right")
    t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1, 1.55)
    for (i, j), c in t.get_celld().items():
        c.set_edgecolor("#dfe3ea")
        if j == 0: c.set_text_props(ha="left")
        if i == 0:
            c.set_text_props(weight="bold", ha="left" if j == 0 else "center")
            c.set_facecolor("white"); c.set_height(c.get_height() * 1.6)
        else:
            c.set_facecolor("#f6f8fb" if i % 2 == 0 else "white")
    plt.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()

## 5. Run the benchmark and show the table
Loops over all models. Results are cached in `results_zeroshot.json` after each model.

**`RERUN = True`** (default) deletes the cache and re-evaluates every model from scratch. Set **`RERUN = False`** only to resume an interrupted run (finished models are skipped). A model that fails is reported and skipped. At the end it saves `results_zeroshot.csv` and `benchmark_table.png`.

In [ ]:
RERUN = True   # True = evaluate everything again | False = resume, skipping finished models

if RERUN:
    RESULTS.unlink(missing_ok=True)
    results = {}
else:
    results = json.loads(RESULTS.read_text()) if RESULTS.exists() else {}

for name, w, cls in MODELS:
    if name in results:
        print(f"skip {name} (done)"); continue
    print(f"\n===== {name} =====")
    try:
        results[name] = run_one(name, w, cls, data_yaml)
    except Exception as e:
        print(f"!! {name} failed: {e}")
        continue
    RESULTS.write_text(json.dumps(results, indent=1))

if not results:
    raise SystemExit("No model finished successfully - check the errors above.")

df = pd.DataFrame(results.values())
df = df.sort_values("mAP@50:95", ascending=False).reset_index(drop=True)
df.to_csv(WORK / "results_zeroshot.csv", index=False)
print(df.to_string(index=False))
render_table(df, str(WORK / "benchmark_table.png"))